# 02 · Preparing knowledge for an application

**Session 01 · notebook 2 of 6**

## Objective

Turn a collection of documents into something an application can search: documents with metadata, split into chunks that keep their meaning, represented as vectors in a vector store. See how the choices made here decide what retrieval can find later.

## The application after this notebook

Diagram: documents, metadata, chunks, embeddings, then the vector store. Below, added in notebook 03: the question and the retrieved context from the vector store feed the prompt, then the model and the answer.

It shows the pipeline this notebook builds, and where notebook 03 picks it up.

The tools are LangChain's: `Document`, `MarkdownHeaderTextSplitter`, `RecursiveCharacterTextSplitter`, `HuggingFaceEmbeddings`, and `InMemoryVectorStore`.

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/rafmacalaba/ai4data.git"   # replace with the published repository when released
BRANCH = "feat/training-session-01-v2"
SESSION = "training/session-01-building-ai-applications-v2/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook replays recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
model = w.make_model()
print("Model:", w.mode(model))

Model: recorded outputs (no WORKSHOP_API_KEY set)


## 1. The collection

Eight short documents, each loaded as a LangChain `Document` whose metadata comes from the file's YAML header: id, title, type, country, date, source. Four are relevant briefs, one is a definition note, one is an outdated summary, and two are distractors. See `assets/README.md` for what each is for.

In [2]:
docs = w.load_documents()
print(f"{'id':20s} {'type':16s} {'date':12s} title")
for d in docs:
    m = d.metadata
    print(f"{m['id']:20s} {m['type']:16s} {str(m['date']):12s} {m['title']}")

id                   type             date         title
coffee               sector note      2026-10-13   Coffee exports in East Africa
internet-indicator   indicator note   2026-10-13   How the internet-users indicator is defined and measured
kenya-digital        country brief    2026-10-13   Kenya: internet use and mobile subscriptions, 2010 to 2024
kenya-economy        country brief    2026-10-13   Kenya: GDP per capita and growth, 2010 to 2024
mobile-money         topic note       2026-10-13   Mobile money in Kenya and Rwanda
regional-2016        summary          2016-06-01   Regional digital summary, 2016
rwanda-digital       country brief    2026-10-13   Rwanda: internet use and mobile subscriptions, 2010 to 2024
rwanda-economy       country brief    2026-10-13   Rwanda: GDP per capita and growth, 2010 to 2024


## 2. Metadata is part of the knowledge

The outdated summary is the clearest case. Its text reads as current ("the latest available figures are for 2015") and its figures were correct when it was written. The only thing that tells an application to rank it below the 2026 briefs is its date, which lives in the metadata.

In [3]:
d = next(d for d in docs if d.metadata["id"] == "regional-2016")
for k in ("date", "type", "note"):
    print(f"{k:6s}", d.metadata[k])
print()
print(d.page_content[:500])

date   2016-06-01
type   summary
note   Written for the training workshop as an example of an outdated document. Not an official publication.

# Regional digital summary, 2016

Internet use in East Africa continues to expand. In Kenya, 16.6 percent of the population used the internet in 2015, up from 7.2 percent in 2010. In Rwanda the share was 12.5 percent in 2015, up from 8.0 percent in 2010.

Mobile cellular subscriptions reached 80.1 per 100 people in Kenya and 75.3 per 100 people in Rwanda in 2015.

On current trends, internet use in both countries is expected to reach about 25 percent of the population by 2020 and about half of t


## 3. Chunking: fixed windows against sections

A chunk is the unit that retrieval returns and the model reads. Two LangChain splitters on the same brief:

- `RecursiveCharacterTextSplitter`: windows of 400 characters with an overlap. Simple, and blind to the document's structure.
- `MarkdownHeaderTextSplitter`: one chunk per heading, with the heading kept in the text and the section name added to the metadata.

Animation: the same Kenya brief cut two ways. Fixed 400-character windows end on a stranded heading and mix two sections; section chunks keep each heading with its paragraph.

*The animation loops. It is drawn from the real brief and the two splitters, so the boundaries are the ones the cell below prints.*

In [4]:
kenya = [d for d in docs if d.metadata["id"] == "kenya-digital"]
fixed = w.split_fixed(kenya, chunk_size=400, chunk_overlap=50)
sections = w.split_sections(kenya)
print(f"fixed windows: {len(fixed)} chunks · sections: {len(sections)} chunks\n")

print("A fixed window (note where it starts and ends):\n")
print(repr(fixed[1].page_content))
print("\nA section chunk:\n")
print(sections[1].page_content)
print("\nmetadata:", sections[1].metadata["section"], "|", sections[1].metadata["date"], "|", sections[1].metadata["country"])

fixed windows: 4 chunks · sections: 4 chunks

A fixed window (note where it starts and ends):

'Over the full period the share increased by about 28 percentage points. The fastest growth in absolute terms came between 2010 and 2015, when the share more than doubled.\n\n## Mobile subscriptions'

A section chunk:

## Mobile subscriptions  
Mobile cellular subscriptions per 100 people stood at 60.0 in 2010, 80.1 in 2015, and 117.6 in 2020. The figure was 121.2 in 2022, 120.6 in 2023, and 126.5 in 2024. Values above 100 reflect multiple subscriptions per person and inactive SIM cards, so the indicator measures subscriptions, and no inference about the share of people with a phone follows from it.

metadata: Mobile subscriptions | 2026-10-13 | Kenya


The fixed window cuts by length, not by meaning. This one starts in the middle of the Internet use section, carries no heading of its own, and ends with the heading of the next section, separated from the paragraph it introduces. A passage about internet use is then stored next to a mobile-subscriptions heading, and nothing in the chunk says which section it belongs to. The section chunk carries its title and heading, which both the retriever and the model can use.

## 4. Embeddings

An embedding is a vector that places a text so that texts with similar meaning are close. The session uses a small sentence-transformer model through LangChain's `HuggingFaceEmbeddings`; it downloads about 90 MB on first use. The cell embeds a few sections and a query and ranks the sections by similarity.

Animation: a chunk of text becomes a list of numbers, then a dot on a map where similar meaning sits close together; a question becomes a star and the two closest chunks are retrieved.

*The animation loops. The similarity scores are the ones the cell below prints; the dot positions and the example numbers are illustrative.*

In [5]:
import numpy as np

chunks = w.split_sections(docs)
embeddings = w.make_embeddings()

sample = [c for c in chunks if c.metadata["section"] in ("Internet use", "Mobile subscriptions", "Growth")][:6]
vectors = np.array(embeddings.embed_documents([c.page_content for c in sample]))
query = np.array(embeddings.embed_query("share of people using the internet"))
print(f"{len(chunks)} chunks · vectors of {vectors.shape[1]} dimensions\n")
for c, v in sorted(zip(sample, vectors), key=lambda cv: -float(cv[1] @ query)):
    print(f"{float(v @ query):6.3f}  {c.metadata['id']:16s} {c.metadata['section']}")

21 chunks · vectors of 384 dimensions

 0.407  kenya-digital    Internet use
 0.398  rwanda-digital   Internet use
 0.290  kenya-digital    Mobile subscriptions
 0.235  rwanda-digital   Mobile subscriptions
 0.045  rwanda-economy   Growth
 0.042  kenya-economy    Growth


Sections about internet use score highest for a query about internet use, the mobile-subscription sections next, and growth last. The ranking depends on what the chunk text contains, which is why the splitter choice in section 3 matters here.

## 5. What poor preparation does

The same collection with the headers stripped, every document concatenated into one text, and fixed windows. Search for the latest Kenya figures and compare the top results with the section index.

In [6]:
from langchain_core.documents import Document

blob = Document(page_content="\n\n".join(d.page_content for d in docs), metadata={"id": "blob", "date": "", "section": "(fixed window)"})
poor = w.build_store(w.split_fixed([blob], chunk_size=400, chunk_overlap=50), embeddings)
good = w.build_store(chunks, embeddings)

q = "What are the latest available figures for internet use in Kenya and Rwanda?"
print("Poorly prepared (one blob, fixed windows, no metadata):")
for d, s in poor.similarity_search_with_score(q, k=3):
    print(f"  {s:6.3f}  {d.page_content[:90]!r}")
print("\nPrepared (sections with metadata):")
w.print_hits(good.similarity_search_with_score(q, k=3))

Poorly prepared (one blob, fixed windows, no metadata):
   0.817  '# Regional digital summary, 2016\n\nInternet use in East Africa continues to expand. In Keny'
   0.782  '# Rwanda: internet use and mobile subscriptions, 2010 to 2024\n\n## Internet use\n\nThe share '
   0.726  '## Reading the two indicators together\n\nMobile subscriptions grew faster than internet use'

Prepared (sections with metadata):
 0.816  [regional-2016 · intro · 2016]
 0.765  [rwanda-digital · Internet use · 2026]
 0.703  [kenya-digital · Internet use · 2026]


In the poorly prepared index the top results are fragments that cannot be cited and carry no date. In the prepared index each hit names its document, section, and year, so the application can filter and the reader can check. The 2016 summary ranks high in the prepared index too; its date is what the next notebook uses to deal with it.

## 6. Build the vector store for the next notebook

Notebook 03 rebuilds this store with one line. `InMemoryVectorStore` is enough for a workshop; the same code runs against a persistent store such as Chroma, FAISS, or pgvector by changing the class.

In [7]:
store = w.build_store(chunks, embeddings)
print(len(store.store), "chunks in the vector store")

21 chunks in the vector store


## Key points

- Metadata (date, type, country, section) is part of what the application knows. It decides what can be filtered and ranked, and what a citation can say.
- Chunk boundaries decide what retrieval can return. Sections keep meaning and headings; fixed windows cut through both.
- Embeddings place text by meaning; the quality of the index is the quality of the text and metadata it was built from.

## Exercise

Write a ninth document, a short brief on Uganda with figures looked up from the World Development Indicators, with the same header fields. Add it to `assets/documents/`, rerun this notebook, and check where its sections rank for the queries above.

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows.
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.